# SEC EDGAR — Escrow & Contingent Share Detection

Pulls 10-K and 10-Q filings for two groups of companies and scans for
escrow / contingent / earnout / founder share disclosures.

**Test group** — companies expected to have contingent share structures:
- CIK 1823584 — Alliance Entertainment (AENT): 60M Class E shares in escrow
- CIK 1821769
- CIK 1830081
- CIK 1979484

**Control group** — large-cap companies that should produce zero escrow findings:
- CIK 320193  — Apple Inc.
- CIK 1652044 — Alphabet Inc. (Google)
- CIK 1045810 — NVIDIA Corporation

All HTTP requests use `User-Agent: jared.goroski@gmail.com` as required by
the SEC EDGAR fair-access policy.  Rate is capped at ~9 req/s (0.11 s delay);
429 responses are retried automatically with exponential back-off.

---
## Imports

In [ ]:
import pandas as pd
from dataclasses import asdict

from sec_filings_sync import fetch_filings_for_ciks
from sec_extractors  import batch_extract_escrow_shares, extract_escrow_shares

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print('Imports OK')

---
## Config

Edit `TEST_CIKS`, `CONTROL_CIKS`, `FORM_TYPES`, or the date range as needed.

In [ ]:
# Companies expected to have contingent / escrow share structures
TEST_CIKS = [
    '1823584',   # Alliance Entertainment Holdings (AENT) — Class E escrow shares
    '1821769',
    '1830081',
    '1979484',
]

# Large-cap control companies — should produce zero escrow findings
CONTROL_CIKS = [
    '320193',    # Apple Inc.
    '1652044',   # Alphabet Inc. (Google)
    '1045810',   # NVIDIA Corporation
]

ALL_CIKS = TEST_CIKS + CONTROL_CIKS

FORM_TYPES = ['10-K', '10-Q']
START_DATE = '2023-01-01'
END_DATE   = '2026-05-05'

# 0.11 s between requests => ~9 req/s, safely under the SEC 10 req/s cap
DELAY = 0.11

print(f'Test CIKs    : {TEST_CIKS}')
print(f'Control CIKs : {CONTROL_CIKS}')
print(f'Form types   : {FORM_TYPES}')
print(f'Date range   : {START_DATE}  ->  {END_DATE}')
print(f'Delay        : {DELAY} s/request')

---
## Step 1 — Fetch filing metadata

`fetch_filings_for_ciks` hits `data.sec.gov/submissions/CIK{cik}.json` for
each company and filters to the requested form types and date range.

In [ ]:
filings = fetch_filings_for_ciks(
    ciks=ALL_CIKS,
    form_types=FORM_TYPES,
    start_date=START_DATE,
    end_date=END_DATE,
    delay=DELAY,
)
print(f'\nTotal filings fetched: {len(filings)}')

In [ ]:
df_filings = pd.DataFrame([asdict(f) for f in filings])

# Counts per company
summary = (
    df_filings.groupby(['cik', 'entity_name', 'ticker'])['form_type']
    .value_counts()
    .rename('count')
    .reset_index()
)
print(summary.to_string(index=False))
print(f'\n{len(filings)} filings total')

---
## Step 2 — Scan all filings for escrow / contingent shares

Each filing is fetched (up to 800 KB), stripped to plain text, then scanned
for trigger keywords.  For each hit a context window is opened and the
nearest share count is extracted using proximity-anchored, placement-verb-
validated matching.

Returns **one row per distinct `(shares, class)` finding** — filings with no
escrow language return zero rows.

| `escrow_type` | Meaning |
|---------------|---------|
| `earnout`     | Earn-out / earnout shares |
| `founder`     | Founder shares subject to forfeiture |
| `performance` | Performance or milestone shares |
| `lockup`      | Lock-up arrangement |
| `contingent`  | SPAC / merger contingent shares |
| `escrow`      | Explicit escrow placement verb, no named type |
| `general`     | Keyword match but no more specific classifier |

In [ ]:
escrow_results = batch_extract_escrow_shares(filings, delay=DELAY)
print(f'\nTotal findings across all filings: {len(escrow_results)}')

---
## Step 3 — Results

### 3a — All findings

In [ ]:
if escrow_results:
    df_escrow = pd.DataFrame([asdict(r) for r in escrow_results])
    df_escrow['shares_in_escrow'] = df_escrow['shares_in_escrow'].apply(lambda x: f'{x:,}')
    display(df_escrow[[
        'entity_name', 'form_type', 'filing_date',
        'share_class', 'shares_in_escrow', 'escrow_type', 'trigger_hint',
    ]])
else:
    print('No escrow findings in this filing set.')

### 3b — Control group verification (Apple / Alphabet / NVIDIA)

In [ ]:
control_names = {
    c.lstrip('0') for c in CONTROL_CIKS
}

if escrow_results:
    df_control = pd.DataFrame([asdict(r) for r in escrow_results])
    df_control = df_control[df_control['cik'].isin(control_names)]
    if df_control.empty:
        print('Control group: no escrow findings (expected)')
    else:
        print(f'WARNING: {len(df_control)} finding(s) in control group — review source_text')
        df_control['shares_in_escrow'] = df_control['shares_in_escrow'].apply(lambda x: f'{x:,}')
        display(df_control[['entity_name', 'form_type', 'filing_date',
                             'share_class', 'shares_in_escrow', 'escrow_type', 'source_text']])
else:
    print('No escrow findings at all.')

---
## Step 4 — Source text review

Print the matched passage for each finding so results can be manually verified.
`source_text` is capped at 500 characters; `trigger_hint` is the extracted
release-condition fragment (up to 300 characters).

In [ ]:
for r in escrow_results:
    print('=' * 80)
    print(f'  {r.entity_name}  |  {r.form_type}  |  {r.filing_date}')
    print(f'  Class       : {r.share_class}')
    print(f'  Shares      : {r.shares_in_escrow:,}')
    print(f'  Type        : {r.escrow_type}')
    print(f'  Trigger     : {r.trigger_hint}')
    print()
    print(r.source_text)
    print()

if not escrow_results:
    print('No findings to review.')